# Crisp method - topological derivative

## 1) Geometry

In [1]:
###############################################################################
## CODE CELL 1 : Import transformer mesh
###############################################################################

from utils.geometry import transformer
from ngsolve.webgui import Draw

mesh = transformer(maxh = 1e-2)                   #  mesh the transformer geometry
print(f"Region names : {mesh.GetMaterials()}")   # display the regions (materials) labels
print(f"Line names : {mesh.GetBoundaries()}")    # display the lines (boundaries) labels
Draw(mesh)

Region names : ('Omega_c', 'Pp', 'Pm', 'Sm', 'Sp')
Line names : ('dOmega', 'dOmega', 'dOmega', 'dOmega', 'dPp', 'dPp', 'dPp', 'dPp', 'dPm', 'dPm', 'dPm', 'dPm', 'dSm', 'dSm', 'dSm', 'dSm', 'dSp', 'dSp', 'dSp', 'dSp')


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## 2) Magnetostatic problem

In [2]:
###############################################################################
## CODE CELL 2 : Define magnetic reluctivity
###############################################################################

from numpy import pi
mu0 = 4e-7 * pi   # magnetic permeability of air (H/m)
mur = 1000        # relative magnetic permeability of the iron    
nu0 = 1/mu0
nuf = nu0/mur

In [3]:
###############################################################################
## CODE CELL 3 : Define the magnetostatic solver
###############################################################################

j = 1e6 # current density in the primary coil (A/m²)

from ngsolve import CF, grad

def curl(v):
    R = CF(((0,1),(-1,0)), dims = (2,2))  # Rotation matrix of angle -pi/2
    return R * grad(v)

from ngsolve import H1, BilinearForm, LinearForm, dx
from utils.solver import solve

def state(nu):
    """ Solve the state equation for a given density field rho """
    fes = H1(mesh, order = 1, dirichlet = "dOmega")
    a, v = fes.TnT()    # define the trial and test functions 
    bf = BilinearForm(curl(v) * (nu * curl(a)) * dx)  
    lf = LinearForm(j * v * dx("Pp") - j * v * dx("Pm"))
    return solve(bf, lf)

#Try it
a,Kinv=state(nu0)
Draw(a)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## 3) Objective function & adjoint problem
### a) Objective function

In [14]:
from utils.solver import flux

# we want to maximize the secondary flux = minimize its opposite
def f(sol):
    return  -flux(sol)

print(f" {f(a) = :.5e} Wb/m")

 f(a) = 2.46138e+00 Wb/m


### b) Adjoint problem

In [15]:
from ngsolve import dx

def df(a, rho, aStar):
    """ Directional derivative of the objective function in the direction aStar """
    return aStar*dx("Sm")  - aStar*dx("Sp")

from utils.optimization import solve_adjoint

p = solve_adjoint(a, nu0,  Kinv, df )
Draw(p)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## 4) Level set to cut ratio

In [16]:
from ngsolve import H1, L2, GridFunction, x,y
V = H1(mesh,order=1)
L = L2(mesh,order=0)
ls = GridFunction(V) #level set function
cr = GridFunction(L) #cut ratio. 1 if element in iron, 0 if in air and inbetween for cut elements
from utils.interpolations import interpolateLevelSetToElems

def nu(cr):
    return (1-cr)*nu0+cr*nuf
    
# Try some level set functions
ls.Set(x*y)
interpolateLevelSetToElems(ls,cr,mesh,"Omega_c") # negative is iron, positive is air
Draw(nu(cr),mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## 5) Topological derivative

In [17]:
###############################################################################
## CODE CELL 6 : Topological derivative
############################################################################### 

from utils.solver import curl
from ngsolve import Integrate,sqrt
import numpy as np

def TD(state,adjoint,nu0,nu1):
    return 2*nu0*(nu1-nu0)/(nu1+nu0)*curl(state)*curl(adjoint) # Topological derivative for changes from nu0 to nu1

def gTD(state,adjoint,cr):
    return (1-cr)*TD(state,adjoint,nu0,nuf)-cr*TD(state,adjoint,nuf,nu0) # generalized topological derivative with negative sign inside the domain
    
def gettheta(ls,g):
    normls=sqrt(Integrate(ls*ls,mesh.Materials("Omega_c")))
    normg=sqrt(Integrate(g*g,mesh.Materials("Omega_c")))
    costheta=min(max(Integrate(g*ls,mesh.Materials("Omega_c"))/normls/normg,-1),1)
    return [np.arccos(costheta),normls,normg] # norms of ls and g and L2 angle between. needed in the update of ls and as break condition

# Test your function
a,Kinv=state(nu(cr))
p=solve_adjoint(a,nu(cr),Kinv,df)
Draw(gTD(a,p,cr),mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## 6) Optimization parameters

In [39]:
# Step size parameters
smax=1
smin=1e-1
sfac=2
sfacplus=1.5
# Break condition
breaktheta=1/180*pi
# Volume weight
M0=0.1
Mall=Integrate(1,mesh.Materials("Omega_c"))
beta=1e-2
lVol=1e-8
def vol(cr):
    return Integrate(cr,mesh.Materials("Omega_c"))/Mall
ls_new=GridFunction(V) # for line search
g=GridFunction(V) # generalized topological derivative, descent direction

## 7) Level set optimization

This algorithm was first published in 

Samuel Amstutz and Heiko Andrae. 2006. A new algorithm for topology optimization using a level-set method. J. Comput. Phys., 216, (August 2006), 573-588. doi: 10.1016/j.jcp.2005.12.015

In [40]:
ls.Set(1) # inital level set function
scene=Draw(-ls,mesh,min=0,max=0)
s=smax # intial step size
interpolateLevelSetToElems(ls,cr,mesh,"Omega_c")
a,Kinv=state(nu(cr))
J=f(a)+beta/2*max(vol(cr)-M0,0)**2 # evaluate initial design
#J=f(a)+lVol*vol(cr) # evaluate initial design
for k in range(1000):
    p=solve_adjoint(a,nu(cr),Kinv,df)
    g.Set(gTD(a,p,cr)+beta*max(vol(cr)-M0,0)/Mall*1) # add derivative of volume
    theta,normls,normg=gettheta(ls,g) 
    print("majiter=",k,"J=",J,"theta=",theta*180/pi)
    if theta<breaktheta: # Break if ls and g are sufficiently close
        break
    for i in range(15): # Line search loop. Reduce step size if no descent achieved
        ls_new.vec.data=(1/np.sin(theta)*(np.sin(theta*(1-s))/normls * ls.vec + np.sin(theta*s)/normg * g.vec))
        interpolateLevelSetToElems(ls_new,cr,mesh,"Omega_c") # Assign new values to cut ratio
        a,Kinv=state(nu(cr))
        Jnew=f(a)+beta/2*max(vol(cr)-M0,0)**2
        print("miniter=",i,"Jnew=",Jnew,"s=",s)
        if Jnew<J: # Line search was succesfull
            s=min(smax,s*sfacplus)
            break
        if s==smin:
            break
        s=max(smin,s/sfac)
    J=Jnew
    ls.vec.data=ls_new.vec
    scene.Redraw()

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

majiter= 0 J= -0.0009678597389152208 theta= 95.76116781456298
miniter= 0 Jnew= -6.90276002970618 s= 1
majiter= 1 J= -6.90276002970618 theta= 91.9971803872336
miniter= 0 Jnew= -0.0009678597389152208 s= 1
miniter= 1 Jnew= -2.9924513591662354 s= 0.5
miniter= 2 Jnew= -5.223978604821971 s= 0.25
miniter= 3 Jnew= -5.9561292079110935 s= 0.125
miniter= 4 Jnew= -6.090307997889122 s= 0.1
majiter= 2 J= -6.090307997889122 theta= 53.328383765168994
miniter= 0 Jnew= -5.829903274283208 s= 0.1
majiter= 3 J= -5.829903274283208 theta= 30.055334702146702
miniter= 0 Jnew= -5.747883790611137 s= 0.1
majiter= 4 J= -5.747883790611137 theta= 27.185639093307092
miniter= 0 Jnew= -5.721027717407434 s= 0.1
majiter= 5 J= -5.721027717407434 theta= 25.688322758347066
miniter= 0 Jnew= -5.703143384077035 s= 0.1
majiter= 6 J= -5.703143384077035 theta= 23.80225939815425
miniter= 0 Jnew= -5.691660621190786 s= 0.1
majiter= 7 J= -5.691660621190786 theta= 21.85164336892044
miniter= 0 Jnew= -5.683886749170786 s= 0.1
majiter= 8

In [ ]:
Mall

## 8) Analysis of the results

### a) Quantities of interest

In [ ]:
u, _ = solveMag(rho2nu(rho),mesh)
print(f"Results BESO : flux = {avgSecondaryFlux(u) : .5e} | vol = {volume(rho)*100 : .3f} % | iterations = {loop}")